### CO2 Emission Predcition Using Different ML algorithms and Comparing them

--- Data Overview & Preprocessing ---

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

df=pd.read_csv("fuel_data.csv")

# data overview
print("first five rows:\n",df.head(5))
print("\nShape of the data\n:",df.shape)
print("\ncolumns of data:\n",df.columns)
print("\nnull values:\n",df.isnull().sum())
df.info()

# DATA PREPROCESSING

fuel_type_dict = {
    "X": "Regular Gasoline",
    "Z": "Premium Gasoline",
    "D": "Diesel",
    "E": "Ethanol (E85)"
}
df["Fuel type"]=df["Fuel type"].replace(fuel_type_dict)

df_encoded = pd.get_dummies(df, columns=["Vehicle class","Transmission","Fuel type"], drop_first=True)

first five rows:
    Model year   Make  ... CO2 rating Smog rating
0        2025  Acura  ...          6           6
1        2025  Acura  ...          6           5
2        2025  Acura  ...          5           5
3        2025  Acura  ...          4           4
4        2025  Acura  ...          4           4

[5 rows x 15 columns]

Shape of the data
: (650, 15)

columns of data:
 Index(['Model year', 'Make', 'Model', 'Vehicle class', 'Engine size (L)',
       'Cylinders', 'Transmission', 'Fuel type', 'City (L/100 km)',
       'Highway (L/100 km)', 'Combined (L/100 km)', 'Combined (mpg)',
       'CO2 emissions (g/km)', 'CO2 rating', 'Smog rating'],
      dtype='object')

null values:
 Model year              0
Make                    0
Model                   0
Vehicle class           0
Engine size (L)         0
Cylinders               0
Transmission            0
Fuel type               0
City (L/100 km)         0
Highway (L/100 km)      0
Combined (L/100 km)     0
Combined (mpg)     

### Linear Regression for CO2 Emission

--- with data leakage ---

In [4]:
X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating"],axis=1)
y=df_encoded["CO2 emissions (g/km)"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)


mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2=r2_score(y_test,y_pred)

print("\n===== WITH DATA LEAKAGE =====\n")
print("Intercept:",model.intercept_)
print("Coefficient:")
for feat,coef in zip(X,model.coef_):
    print(f"{feat}:{coef:.2f}")

print("\nMSE:",mse)
print("\nMAE:",mae)
print("\nR2 SCORE:",r2)
print("RMSE:", rmse)


===== WITH DATA LEAKAGE =====

Intercept: 40.43862347398277
Coefficient:
Model year:-0.00
Engine size (L):0.28
Cylinders:-0.53
City (L/100 km):6.25
Highway (L/100 km):4.81
Combined (L/100 km):12.28
Combined (mpg):-0.09
Vehicle class_Full-size:-0.49
Vehicle class_Mid-size:0.64
Vehicle class_Minicompact:-2.23
Vehicle class_Minivan:-0.19
Vehicle class_Pickup truck: Small:0.47
Vehicle class_Pickup truck: Standard:1.04
Vehicle class_Sport utility vehicle: Small:0.24
Vehicle class_Sport utility vehicle: Standard:0.42
Vehicle class_Station wagon: Mid-size:1.70
Vehicle class_Station wagon: Small:-0.03
Vehicle class_Subcompact:-0.44
Vehicle class_Two-seater:-0.17
Transmission_A6:-0.60
Transmission_A8:0.72
Transmission_A9:-0.15
Transmission_AM6:1.91
Transmission_AM7:0.17
Transmission_AM8:-0.20
Transmission_AS10:-0.03
Transmission_AS6:0.03
Transmission_AS8:-0.30
Transmission_AS9:0.59
Transmission_AV:0.57
Transmission_AV1:0.30
Transmission_AV10:0.33
Transmission_AV6:-0.54
Transmission_AV7:-0.19
T

by looking at the r2 score and some unrealistic soefficient values i found out there is some data leakage. 
I included "City (L/100 km)","Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)" in the features
But CO₂ emissions are directly calculated from fuel consumption

### Linear Regression for CO2 Emission

--- without data leakage ---

In [5]:
X=df_encoded.drop(columns=["Make","Model","Smog rating","CO2 emissions (g/km)","CO2 rating","City (L/100 km)",
                           "Highway (L/100 km)","Combined (L/100 km)","Combined (mpg)"],axis=1)
y=df_encoded["CO2 emissions (g/km)"]

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)
r2=r2_score(y_test,y_pred)


In [6]:
print("\n===== WITHOUT DATA LEAKAGE =====\n")
print("Intercept:",model.intercept_)
print("Coefficient:")
for feat,coef in zip(X,model.coef_):
    print(f"{feat}:{coef:.2f}")


===== WITHOUT DATA LEAKAGE =====

Intercept: 119.92692824598365
Coefficient:
Model year:-0.00
Engine size (L):19.12
Cylinders:11.28
Vehicle class_Full-size:-6.44
Vehicle class_Mid-size:-6.15
Vehicle class_Minicompact:7.15
Vehicle class_Minivan:4.51
Vehicle class_Pickup truck: Small:28.57
Vehicle class_Pickup truck: Standard:41.26
Vehicle class_Sport utility vehicle: Small:18.27
Vehicle class_Sport utility vehicle: Standard:26.48
Vehicle class_Station wagon: Mid-size:44.01
Vehicle class_Station wagon: Small:24.25
Vehicle class_Subcompact:2.93
Vehicle class_Two-seater:13.79
Transmission_A6:-0.25
Transmission_A8:39.78
Transmission_A9:21.11
Transmission_AM6:-59.97
Transmission_AM7:7.59
Transmission_AM8:29.10
Transmission_AS10:26.10
Transmission_AS6:-7.73
Transmission_AS8:8.70
Transmission_AS9:14.06
Transmission_AV:-41.65
Transmission_AV1:-20.15
Transmission_AV10:-34.33
Transmission_AV6:-49.05
Transmission_AV7:-16.17
Transmission_AV8:-3.54
Transmission_M5:0.00
Transmission_M6:25.47
Transmi

In [7]:
print("\nMSE:",mse)
print("\nMAE:",mae)
print("\nR2 SCORE:",r2)
print("RMSE:", rmse)


MSE: 783.8103073294116

MAE: 22.123549666004145

R2 SCORE: 0.80672547614114
RMSE: 27.99661242595989


### Random Forest Regressor

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf_model=RandomForestRegressor()

param_grid={
    'n_estimators':[100,200],
    'max_depth':[None,10,20],
    'min_samples_split':[2,3,5],
    'min_samples_leaf':[1,2,3],
    'random_state':[42]
}
grid=GridSearchCV(rf_model,param_grid,cv=5)
grid.fit(X_train,y_train)
best_rf = grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
print(grid.best_params_)
print("The co2 emission prediction is:",y_pred_rf)


{'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200, 'random_state': 42}
The co2 emission prediction is: [188.15083333 311.18641667 189.5484697  219.19821151 281.40632612
 276.677875   187.7264881  252.97005713 221.82923413 337.53470274
 170.14025397 180.175875   252.30083532 297.37530952 203.75821343
 252.30083532 268.77034957 217.38455952 277.70354167 210.51783333
 228.15870779 215.47083333 238.81521735 260.58835155 327.3115
 256.3895     185.87268452 287.40976587 202.24047619 222.75481835
 178.42957143 332.60083694 337.53470274 293.58233333 291.30913837
 148.09258333 198.2179881  131.37999745 148.09258333 277.65068117
 346.87303571 124.3503869  327.40991667 247.31866966 352.94558081
 222.19132143 264.39108431 194.2965     173.22325    249.07886021
 236.71992857 224.96970094 308.76901389 124.3503869  222.75481835
 261.1506461  185.87268452 189.5484697  281.40632612 325.95539646
 247.31866966 352.94558081 317.583151   209.91026281 241.40658333
 380.5

In [9]:
MSE=mean_squared_error(y_test,y_pred_rf)
MAE=mean_absolute_error(y_test,y_pred_rf)
R2=r2_score(y_test,y_pred_rf)
print("MSE:\n",MSE)
print("MAE:\n",MAE)
print("R2_score:\n",R2)

MSE:
 574.3956173562299
MAE:
 17.92515449863141
R2_score:
 0.8583636392466008


### Decision Tree Regressor

In [10]:
from sklearn.tree import DecisionTreeRegressor

dt_model=DecisionTreeRegressor()
dt_model.fit(X_train,y_train)
y_pred_dt=dt_model.predict(X_test)

MSE=mean_squared_error(y_test,y_pred_dt)
MAE=mean_absolute_error(y_test,y_pred_dt)
R2=r2_score(y_test,y_pred_dt)
print("The co2 emission prediction is:",y_pred_dt)


The co2 emission prediction is: [188.         308.5        191.         222.         296.
 287.         185.         271.         239.         338.33333333
 161.         180.         222.         296.66666667 204.14285714
 222.         261.         218.5        293.5        205.
 228.5        216.         226.         251.         334.
 254.         181.         287.25       210.         222.6
 174.         333.         338.33333333 280.         284.
 144.5        204.         132.2        144.5        278.5
 347.66666667 124.33333333 354.         250.5        353.
 224.         267.42857143 194.66666667 178.5        226.
 266.         225.4        307.66666667 124.33333333 222.6
 255.         181.         191.         296.         327.
 250.5        353.         318.33333333 217.         242.
 387.         144.5        379.         327.         210.
 318.33333333 248.         263.         178.5        270.
 336.         244.71428571 124.33333333 232.         119.5
 354.         237.5 

In [11]:
print("MSE:\n",MSE)
print("MAE:\n",MAE)
print("R2_score:\n",R2)

MSE:
 804.3731210971566
MAE:
 19.017051282051284
R2_score:
 0.8016550298826564


### Kneighbours Regressor

In [14]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe=Pipeline([('scaler',StandardScaler()),
               ('model',KNeighborsRegressor())
               ])

param_grid_kr={
    'model__n_neighbors':[3,5,7,9],
    'model__weights':['uniform','distance']
}

grid=GridSearchCV(pipe,param_grid_kr,cv=5,scoring='r2')
grid.fit(X_train,y_train)

best_kr=grid.best_estimator_
y_pred_kr=best_kr.predict(X_test)

MSE=mean_squared_error(y_test,y_pred_kr)
MAE=mean_absolute_error(y_test,y_pred_kr)
R2=r2_score(y_test,y_pred_kr)
print(grid.best_params_)
print("The co2 emission prediction is:",y_pred_kr)

{'model__n_neighbors': 7, 'model__weights': 'distance'}
The co2 emission prediction is: [221.82709809 308.49999786 191.         222.         295.99999106
 287.         185.         271.         226.15793271 338.33333333
 161.03487983 144.76667594 247.4        296.66666132 207.
 247.4        261.         218.5        272.53853533 205.0000137
 228.5        216.         247.87813119 251.         334.
 253.         233.09639076 287.25       209.99999021 213.14285714
 195.36963838 333.         338.33333333 280.         284.00000382
 144.5        204.00001061 132.20000431 144.5        275.71428571
 351.22539541 124.33333333 354.         250.5        352.99999529
 224.         267.42857143 194.66666667 178.5        265.90835565
 226.89893377 225.4        307.66666667 124.33333333 213.14285714
 286.40973265 182.80506701 191.         295.99999106 327.
 250.5        352.99999529 304.71428571 217.         242.
 386.99997777 144.5        379.         327.         194.80312598
 304.71428571 238.182

In [15]:
print("MSE:\n",MSE)
print("MAE:\n",MAE)
print("R2_score:\n",R2)

MSE:
 929.6627639791677
MAE:
 21.530005630550477
R2_score:
 0.7707606976111487
